# SmolVLA の推論サーバー(Colab の GPU で動かし、PC の Gazebo 評価から呼ぶ)
PC(ノート、GPU なし)での CPU 推論は 1 回 約55秒で、評価が回らない。ここで GPU 推論サーバーを立て、トンネル(cloudflared)経由で PC の `eval_smolvla.bat` から使う。
**ランタイム → ランタイムのタイプを変更 → GPU(L4 か T4 で足りる)** にしてから、上から順に実行する。
- 通信は トークン付きの HTTPS。URL とトークンはこのノートブックを開いている人だけに表示される(他人に見せない)。
- セル6 を実行している間だけサーバーが生きる。評価が終わったら止める。タブは開いたままにすること。

In [ ]:
# 1) Drive をつなぎ、チェックポイント(final)を探す
from google.colab import drive
drive.mount('/content/drive')
import os, glob, sys, shutil, subprocess, json, time, secrets
def sh(cmd):
    print('$', cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    p.wait()
    assert p.returncode == 0, '失敗(終了コード %d): %s' % (p.returncode, cmd)
NAME = 's2_mix_1025_smolvla'                         # 学習で作ったモデルの名前(tools\names.conf の S2_SMOLVLA の、/final の手前)。ここだけ書き換える
CKPT = None                                          # 自動で見つからないときだけ、ここに final のフォルダを直接書く
if CKPT is None:
    hits = sorted(glob.glob('/content/drive/MyDrive/checkpoints/train/' + NAME + '/final') + glob.glob('/content/drive/MyDrive/*/checkpoints/train/' + NAME + '/final') + glob.glob('/content/drive/MyDrive/*/*/checkpoints/train/' + NAME + '/final'))
    print('見つかった候補:', hits)
    assert hits, 'final が Drive に見つからない'
    CKPT = hits[0]
print('モデル名:', NAME)
print('CKPT =', CKPT, os.listdir(CKPT))

In [ ]:
# 2) GPU の確認
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
print(sys.version)

In [ ]:
# 3) lerobot 0.3.3(SmolVLA つき)を入れる(5〜10分)。学習のときと同じ版。※ 先に「ランタイム → セッションを再起動」しておくこと
sh('pip install -q torch==2.7.0 torchvision==0.22.0 torchcodec==0.4.0')
sh('pip install -q --only-binary=:all: "av>=14.2.0" datasets==3.6.0')
sh('pip install -q "lerobot[smolvla]==0.3.3"')
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy; print('SmolVLA import OK')

In [ ]:
# 4) 推論サーバーのプログラムを書き出す(PC 側の tools/smolvla_http_server.py と同じもの)
SRV = r'''# SmolVLA の推論サーバー(HTTP 版。Colab の GPU で動かし、トンネル経由で PC の評価から呼ぶ。lerobot 0.3.3)。
# eval_act.py(--smolvla)が、画像・状態・言語指示を送り、行動の塊(50コマ×5関節、正規化を戻した値)を受け取る。
# usage: smolvla_http_server.py --model-path DIR --token TOKEN [--port 8000] [--device auto] [--flow-steps N]
# 要求: POST /predict (ヘッダ X-Token が一致しないと拒否)。本文は JSON(画像は base64)。pickle は使わない
import argparse, base64, json, os, shutil, sys, tempfile, time, traceback
import numpy as np
import torch
from http.server import BaseHTTPRequestHandler, HTTPServer


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--model-path', required=True)
    ap.add_argument('--port', type=int, default=8000)
    ap.add_argument('--token', required=True)
    ap.add_argument('--device', default='auto')
    ap.add_argument('--flow-steps', type=int, default=0, help='0=学習時の設定(10)のまま。減らすと速いが精度が変わる')
    a = ap.parse_args()
    dev = ('cuda' if torch.cuda.is_available() else 'cpu') if a.device == 'auto' else a.device
    torch.set_num_threads(max(1, os.cpu_count() or 1))
    # 学習側の config.json は device が cuda のことがあるので、書き換えた一時コピーから読む
    tmp = os.path.join(tempfile.gettempdir(), 'smolvla_ckpt_' + os.path.basename(os.path.normpath(a.model_path)))
    shutil.rmtree(tmp, ignore_errors=True); shutil.copytree(a.model_path, tmp)
    cp = os.path.join(tmp, 'config.json'); cfg = json.load(open(cp)); cfg['device'] = dev
    json.dump(cfg, open(cp, 'w'), indent=2)
    from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
    t0 = time.time()
    pol = SmolVLAPolicy.from_pretrained(tmp).to(dev).eval()
    if a.flow_steps > 0:
        pol.config.num_steps = a.flow_steps
        try: pol.model.config.num_steps = a.flow_steps
        except Exception: pass
    c = pol.config
    img_key = next(iter(c.image_features.keys()))
    print('読み込み %.1f 秒 / device=%s / 画像キー=%s / 入力=%s / 出力=%s / chunk=%d / 流れのステップ数=%s' % (
        time.time() - t0, dev, list(c.image_features.keys()), {k: tuple(v.shape) for k, v in c.input_features.items()},
        {k: tuple(v.shape) for k, v in c.output_features.items()}, c.chunk_size, c.num_steps), flush=True)
    # 正規化の統計が入っているか(平均0・標準偏差1のままなら、学習時の統計が読めていない)
    for n_, b in pol.named_buffers():
        if n_.endswith('mean') or n_.endswith('std'):
            print('  正規化 %s: %s' % (n_, np.round(b.detach().cpu().numpy().flatten()[:6], 3).tolist()), flush=True)

    def infer(img_bytes, h, w, state, task, seed=None):
        img = torch.from_numpy(np.frombuffer(img_bytes, np.uint8).reshape(h, w, 3).copy()).permute(2, 0, 1).float().div(255.0).unsqueeze(0)
        batch = {img_key: img.to(dev), 'observation.state': torch.tensor([state], dtype=torch.float32).to(dev), 'task': [task]}
        if seed is not None:                                  # 流れの出発点の雑音を決まったものにする(再試で同じ入力なら同じ出力)
            torch.manual_seed(int(seed))
            if torch.cuda.is_available(): torch.cuda.manual_seed_all(int(seed))
        with torch.inference_mode():
            return pol.predict_action_chunk(batch).detach().cpu().numpy()[0].astype(np.float32)

    infer(np.zeros((240, 320, 3), np.uint8).tobytes(), 240, 320, [0.0] * c.input_features['observation.state'].shape[0], 'put the red block in the box')    # ならし運転
    t1 = time.time(); ch = infer(np.zeros((240, 320, 3), np.uint8).tobytes(), 240, 320, [0.0] * c.input_features['observation.state'].shape[0], 'put the red block in the box')
    print('ならし運転: 1回の推論 %.2f 秒 / 出力 %s' % (time.time() - t1, ch.shape), flush=True)
    state_dim = c.input_features['observation.state'].shape[0]
    first = [True]

    class H(BaseHTTPRequestHandler):
        def log_message(self, *args): pass
        def _send(self, code, obj):
            b = json.dumps(obj).encode()
            self.send_response(code); self.send_header('Content-Type', 'application/json'); self.send_header('Content-Length', str(len(b))); self.end_headers(); self.wfile.write(b)
        def do_GET(self):
            self._send(200, dict(ok=True, device=dev, state_dim=state_dim, chunk=int(c.chunk_size)))
        def do_POST(self):
            if self.headers.get('X-Token') != a.token: return self._send(403, dict(err='bad token'))
            try:
                req = json.loads(self.rfile.read(int(self.headers.get('Content-Length', 0))))
                t = time.time()
                ch = infer(base64.b64decode(req['img']), req['h'], req['w'], req['state'], req['task'], req.get('seed'))
                if first[0]:
                    print('最初の要求: 状態=%s 出力の先頭=%s 末尾=%s (%.2f 秒)' % (np.round(req['state'], 3).tolist(), np.round(ch[0], 3).tolist(), np.round(ch[-1], 3).tolist(), time.time() - t), flush=True); first[0] = False
                self._send(200, dict(actions=base64.b64encode(ch.tobytes()).decode(), n=int(ch.shape[0]), dim=int(ch.shape[1]), sec=time.time() - t))
            except Exception:
                traceback.print_exc(); self._send(500, dict(err=traceback.format_exc()[-400:]))

    srv = HTTPServer(('127.0.0.1', a.port), H)
    print('READY', flush=True)
    srv.serve_forever()


if __name__ == '__main__':
    main()
'''
open('/content/smolvla_http_server.py', 'w', encoding='utf-8').write(SRV)
print('書き出しました', len(SRV), '文字')

In [ ]:
# 5) サーバーを起動し、トンネルをつなぐ(モデルの読み込みに1〜2分)
import re
TOKEN = secrets.token_urlsafe(24)
subprocess.run('pkill -f "[s]molvla_http_server.py"; pkill -f "[c]loudflared"; sleep 1', shell=True)
srv_log = open('/content/server.log', 'w')
srv = subprocess.Popen([sys.executable, '/content/smolvla_http_server.py', '--model-path', CKPT, '--token', TOKEN, '--port', '8000'], stdout=srv_log, stderr=subprocess.STDOUT)
for i in range(120):
    time.sleep(3)
    txt = open('/content/server.log').read()
    if 'READY' in txt: break
    assert srv.poll() is None, 'サーバーが落ちました:\n' + txt[-3000:]
print(open('/content/server.log').read()[-2500:])
assert 'READY' in open('/content/server.log').read(), '準備できません'
if not os.path.exists('/content/cloudflared'):
    sh('wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared')
tun_log = open('/content/tunnel.log', 'w')
tun = subprocess.Popen(['/content/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'], stdout=tun_log, stderr=subprocess.STDOUT)
URL = None
for i in range(40):
    time.sleep(2)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunnel.log').read())
    if m: URL = m.group(0); break
assert URL, 'トンネルの URL が取れません:\n' + open('/content/tunnel.log').read()[-2000:]
time.sleep(5)
import urllib.request
print('疎通確認:', urllib.request.urlopen(urllib.request.Request(URL + '/', headers={'User-Agent': 'pai-eval'}), timeout=30).read().decode())
print()
print('=== PC で使う値(PC の pai_tools フォルダで、下のコマンドを実行する) ===')
print('URL   =', URL)
print('TOKEN =', TOKEN)
print('stage2_eval.bat vla  %s %s' % (URL, TOKEN))
print('stage2_eval.bat seen %s %s' % (URL, TOKEN))

In [ ]:
# 6) 評価の間、このセルを動かしたままにする(サーバーのログを時々表示)。評価が終わったら、このセルを停止(■)する
try:
    while True:
        time.sleep(60)
        assert srv.poll() is None, 'サーバーが落ちました'
        ok = 'OK'
        try: urllib.request.urlopen(urllib.request.Request(URL + '/', headers={'User-Agent': 'pai-eval'}), timeout=20).read()
        except Exception as e: ok = 'NG(トンネルに届かない: %s)→ 評価を止め、このノートブックを再実行して URL と TOKEN を取り直す' % str(e)[:60]
        print(time.strftime('%H:%M:%S'), 'トンネル', ok, '|', open('/content/server.log').read().strip().splitlines()[-1][:100])
except KeyboardInterrupt:
    pass
finally:
    srv.terminate(); tun.terminate(); print('停止しました')

In [ ]:
print(open('/content/server.log').read()[-3000:])